# 03 - Feature Engineering

## 1. Introduction and prediction objective

This notebook turns the cleaned weekly dengue counts into modelling datasets.

**Prediction setup:** for each district, standing at week *t*, predict week *t + 1*. All input features use only week *t* and earlier weeks; the targets come from week *t + 1*.

**Targets**
- `cases_next` - next week's case count (regression, original units)
- `target_log_next` - `log1p` of next week's case count (regression, log scale)
- `risk_next` - next week's risk level: low / medium / high (classification)

**Input features (11):** current cases, three weekly lags, a 52-week seasonal lag, 4- and 8-week moving averages, the weekly change, two seasonal encodings and the district code. All case-based features are kept in actual case units (no log transform), because tree-based models such as Random Forest, XGBoost and LightGBM do not need transformed or scaled inputs, and raw counts are easier to interpret.

**Input:** `dataset/processed/02_cleaned_weekly_data.csv`
**Output:** `dataset/processed/train.csv`, `dataset/processed/test.csv`, `dataset/processed/03_risk_thresholds.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../dataset")
IN_PATH = DATA_DIR / "processed" / "02_cleaned_weekly_data.csv"
TRAIN_PATH = DATA_DIR / "processed" / "train.csv"
TEST_PATH = DATA_DIR / "processed" / "test.csv"
THRESH_PATH = DATA_DIR / "processed" / "03_risk_thresholds.csv"

TRAIN_END = pd.Timestamp("2023-12-31")  # training targets up to this date, test targets after it

## 2. Load cleaned data

After notebook 02 every district has one row per week, and consecutive weeks are exactly 7 days apart. This means "*k* observations earlier" is the same as "*k* weeks earlier", which the historical features below rely on.

`district_code` was created in notebook 02. It is a **categorical location identifier** (districts numbered alphabetically). The number itself has no meaning; it only lets the model learn district-specific patterns, which matters because case levels differ widely between districts (e.g. Colombo vs Mullaitivu).

In [2]:
cleaned = pd.read_csv(IN_PATH, parse_dates=["start_date", "end_date"])
df = cleaned.sort_values(["district", "start_date"]).reset_index(drop=True)

print(df.shape)
print("districts:", df["district"].nunique(), "| weeks per district:", df.groupby("district").size().unique())
print("weeks:", df["start_date"].min().date(), "to", df["start_date"].max().date())

(26728, 10)
districts: 26 | weeks per district: [1028]
weeks: 2006-12-23 to 2026-08-29


## 3. Current observations and historical features

**Terminology:** current observation and **lag features** (first-, second- and third-order lags, plus an annual seasonal lag).

| Feature | Meaning |
|---|---|
| `current_week_cases` | cases in the current week *t* |
| `previous_week_cases` | cases in week *t - 1* |
| `previous_2_weeks_cases` | cases in week *t - 2* |
| `previous_3_weeks_cases` | cases in week *t - 3* |
| `seasonal_cases_previous_52_weeks` | cases 52 weeks earlier (about one year ago) |

**Why they help:** dengue counts are strongly autocorrelated - a district with many cases this week usually has many next week. The last few weeks also show whether an outbreak is building up or dying down. The 52-week lag captures the yearly pattern linked to the monsoon seasons.

**How:** the rows are sorted by date within each district, and each column is shifted down by *k* rows **within each district** (`groupby("district").shift(k)`), so values never move from one district into another. The first *k* weeks of each district have no earlier value and are left empty (handled in section 8).

In [3]:
district_cases = df.groupby("district")["cases"]

df["current_week_cases"] = df["cases"]
df["previous_week_cases"] = district_cases.shift(1)
df["previous_2_weeks_cases"] = district_cases.shift(2)
df["previous_3_weeks_cases"] = district_cases.shift(3)
df["seasonal_cases_previous_52_weeks"] = district_cases.shift(52)

LAG_COLS = ["previous_week_cases", "previous_2_weeks_cases", "previous_3_weeks_cases",
            "seasonal_cases_previous_52_weeks"]

colombo = df[df["district"] == "Colombo"].set_index("start_date")
colombo[["current_week_cases", *LAG_COLS]].iloc[52:57]

,current_week_cases,previous_week_cases,previous_2_weeks_cases,previous_3_weeks_cases,seasonal_cases_previous_52_weeks
start_date,,,,,
2007-12-22,40,44.0,80.0,71.0,71.0
2007-12-29,49,40.0,44.0,80.0,40.0
2008-01-05,44,49.0,40.0,44.0,43.0
2008-01-12,23,44.0,49.0,40.0,38.0
2008-01-19,38,23.0,44.0,49.0,52.0


## 4. Moving averages and weekly changes

**Terminology:** **rolling (moving) averages** and a **first-order difference**.

| Feature | Meaning |
|---|---|
| `moving_average_4_weeks` | mean of weeks *t-3* to *t* (4 weeks) |
| `moving_average_8_weeks` | mean of weeks *t-7* to *t* (8 weeks) |
| `weekly_case_change` | `current_week_cases - previous_week_cases` |

**Why they help:** weekly counts are noisy (late reports, holidays). Moving averages give a smoother short-term (4-week) and medium-term (8-week) level of transmission. The weekly change shows the direction and speed of the trend: a large positive value can be an early sign of an outbreak.

**How:** `rolling(window=k).mean()` is applied within each district. The window includes the current week and the previous *k - 1* weeks only, never future weeks. A full window is required, so the first *k - 1* weeks of each district are empty.

In [4]:
df["moving_average_4_weeks"] = district_cases.transform(lambda x: x.rolling(window=4).mean())
df["moving_average_8_weeks"] = district_cases.transform(lambda x: x.rolling(window=8).mean())
df["weekly_case_change"] = df["current_week_cases"] - df["previous_week_cases"]

colombo = df[df["district"] == "Colombo"].set_index("start_date")
colombo[["current_week_cases", "moving_average_4_weeks", "moving_average_8_weeks", "weekly_case_change"]].iloc[52:57].round(2)

,current_week_cases,moving_average_4_weeks,moving_average_8_weeks,weekly_case_change
start_date,,,,
2007-12-22,40,58.75,53.25,-4.0
2007-12-29,49,53.25,53.88,9.0
2008-01-05,44,44.25,52.62,-5.0
2008-01-12,23,39.00,49.38,-21.0
2008-01-19,38,38.50,48.62,15.0


## 5. Annual seasonal encoding

**Terminology:** **cyclical (sine/cosine) encoding** of the position in the year.

**What they represent:** where the week falls in the calendar year. They describe *calendar* seasonality only - they are **not** weather measurements (no rainfall or temperature data is used in this project).

**Why they help:** dengue in Sri Lanka rises with the monsoon seasons, so the time of year carries information. A plain week number would put week 52 and week 1 far apart (52 vs 1) even though they are neighbours. Mapping the year onto a circle with sine and cosine keeps late December and early January close together.

**How (and ISO week 53):** ISO week numbers run 1-52 in most years but reach 53 in some (e.g. 2015, 2020, 2026). Dividing the ISO week by 52 would push week 53 past a full circle, so the cycle would jump at those year ends. Instead I use the **normalised day of the year** of each week's start date:

`year_position = (day_of_year - 1) / days_in_year` (365 or 366)

This runs from 0 on 1 January to just under 1 on 31 December in every year, so the cycle stays continuous across years and week 53 needs no special treatment. The two features are `sin(2π · year_position)` and `cos(2π · year_position)`.

In [5]:
day_of_year = df["start_date"].dt.dayofyear
days_in_year = np.where(df["start_date"].dt.is_leap_year, 366, 365)
year_position = (day_of_year - 1) / days_in_year

df["seasonal_week_sin"] = np.sin(2 * np.pi * year_position)
df["seasonal_week_cos"] = np.cos(2 * np.pi * year_position)

# around the 2020/21 year end (2020 has an ISO week 53)
colombo = df[df["district"] == "Colombo"].set_index("start_date")
year_end = colombo.loc["2020-12-12":"2021-01-16", ["seasonal_week_sin", "seasonal_week_cos"]].copy()
year_end.insert(0, "iso_week", year_end.index.isocalendar().week)
year_end.round(3)

,iso_week,seasonal_week_sin,seasonal_week_cos
start_date,,,
2020-12-12,50,-0.337,0.942
2020-12-19,51,-0.221,0.975
2020-12-26,52,-0.103,0.995
2021-01-02,53,0.017,1.000
2021-01-09,1,0.137,0.991
2021-01-16,2,0.255,0.967


The ISO week jumps 52 → 53 → 1, but the sine and cosine values change smoothly from one week to the next.

## 6. Next-week regression targets

**Terminology:** **one-step-ahead forecasting targets**.

- `target_date` - the start date of the week being predicted (the next row of the same district)
- `cases_next` - actual cases in that week (regression target in original units)
- `target_log_next` - `log1p(cases_next)` (regression target on a log scale)
- `target_imputed` - whether the target week's value was estimated during cleaning (data-quality flag)

**Why two regression targets:** the counts are very skewed (many weeks near zero, outbreak weeks in the thousands). A log target reduces the influence of a few extreme weeks, but it is not guaranteed to be better - both versions are kept so the modelling notebook can compare them. Predictions from a model trained on `target_log_next` are converted back to approximate case counts with:

```python
predicted_cases = np.expm1(predicted_log_cases)
```

**How:** the next row of each district is taken with `groupby("district").shift(-1)`. Only the targets look forward; none of the input features do.

In [6]:
by_district = df.groupby("district")

df["target_date"] = by_district["start_date"].shift(-1)
df["cases_next"] = by_district["cases"].shift(-1)
df["target_log_next"] = np.log1p(df["cases_next"])
df["target_imputed"] = by_district["is_imputed"].shift(-1)

colombo = df[df["district"] == "Colombo"].set_index("start_date")
colombo[["current_week_cases", "target_date", "cases_next", "target_log_next"]].iloc[52:57].round({"target_log_next": 2})

,current_week_cases,target_date,cases_next,target_log_next
start_date,,,,
2007-12-22,40,2007-12-29,49.0,3.91
2007-12-29,49,2008-01-05,44.0,3.81
2008-01-05,44,2008-01-12,23.0,3.18
2008-01-12,23,2008-01-19,38.0,3.66
2008-01-19,38,2008-01-26,38.0,3.66


## 7. District-specific risk classification

**Terminology:** a **classification target** created by **percentile-based discretisation** of next week's cases.

A single national cut-off would not work: Colombo has hundreds of cases in a normal week, while Mullaitivu usually has 0-5. So each district gets its own thresholds:

| `risk_next` | Rule (for that district) |
|---|---|
| low | `cases_next` at or below the 75th percentile |
| medium | above the 75th and at or below the 90th percentile |
| high | above the 90th percentile |

The percentiles are calculated from **training-period weeks only** (start date up to 31 December 2023), so no information from the test period is used to define the labels.

**Note:** these are *statistically defined, relative* risk categories ("unusually high for this district"), **not** official public-health outbreak or epidemic thresholds.

In [7]:
train_period = df[df["start_date"] <= TRAIN_END]

thresholds = (
    train_period.groupby("district")["cases"]
    .quantile([0.75, 0.90])
    .unstack()
    .set_axis(["medium_above", "high_above"], axis=1)
)
print("thresholds use weeks up to:", train_period["start_date"].max().date())
thresholds.sort_values("high_above")

thresholds use weeks up to: 2023-12-30


,medium_above,high_above
district,,
Mullaitivu,2.0,5.0
Kilinochchi,2.0,6.0
Ampara,4.0,9.0
Mannar,3.0,10.0
NuwaraEliya,6.0,11.0
Polonnaruwa,8.0,14.0
Vavuniya,5.0,15.0
Monaragala,9.0,20.0
Anuradhapura,13.0,26.0


In [8]:
limits = df[["district"]].join(thresholds, on="district")

df["risk_next"] = np.select(
    [df["cases_next"] > limits["high_above"], df["cases_next"] > limits["medium_above"]],
    ["high", "medium"],
    default="low",
)
df["risk_next"] = df["risk_next"].where(df["cases_next"].notna())
df["risk_next"].value_counts()

risk_next
low       19489
medium     4379
high       2834
Name: count, dtype: int64

## 8. Missing-value handling

The historical features are empty where there is not enough history:
- the first 52 weeks of each district have no `seasonal_cases_previous_52_weeks` (the longest look-back),
- the first 3-7 weeks also lack the shorter lags and moving averages.

The last week of each district has no next week, so its targets are empty.

These gaps are expected. Filling them would mean inventing history or using later values, so these rows are **removed** instead. The cleaned case counts themselves are not changed. The data-quality flags `is_imputed`, `dup_week` and `target_imputed` are kept so the modelling stage can exclude unreliable weeks when evaluating.

In [9]:
FEATURES = [
    "current_week_cases",
    "previous_week_cases",
    "previous_2_weeks_cases",
    "previous_3_weeks_cases",
    "seasonal_cases_previous_52_weeks",
    "moving_average_4_weeks",
    "moving_average_8_weeks",
    "weekly_case_change",
    "seasonal_week_sin",
    "seasonal_week_cos",
    "district_code",
]
TARGETS = ["cases_next", "target_log_next", "risk_next"]

print("missing values before removal:")
print(df[FEATURES + TARGETS].isna().sum().loc[lambda s: s > 0])

missing values before removal:
previous_week_cases                   26
previous_2_weeks_cases                52
previous_3_weeks_cases                78
seasonal_cases_previous_52_weeks    1352
moving_average_4_weeks                78
moving_average_8_weeks               182
weekly_case_change                    26
cases_next                            26
target_log_next                       26
risk_next                             26
dtype: int64


In [10]:
before = len(df)
df = df.dropna(subset=FEATURES + TARGETS).reset_index(drop=True)

# whole-number columns went to float because of the gaps - back to integer counts
int_cols = LAG_COLS + ["weekly_case_change", "cases_next"]
df[int_cols] = df[int_cols].astype(int)
df["target_imputed"] = df["target_imputed"].astype(bool)

print("removed", before - len(df), "rows ->", df.shape)
print("per district:", df.groupby("district").size().unique())

removed 1378 rows -> (25350, 25)
per district: [975]


## 9. Chronological train/test split

The split uses the **target date** (the week being predicted), not the input week:
- **train:** target week up to 31 December 2023
- **test:** target week from 1 January 2024

Splitting on the input week would put the row "input 30 Dec 2023 → target 6 Jan 2024" in training, so the model would be trained on a 2024 outcome. Using `target_date` avoids that. The input weeks of the first test rows still come from late 2023, which is fine, because past data is always known at prediction time.

The rows are **not shuffled**: this is a time series, so the model must be trained on the past and tested on the future.

In [11]:
train = df[df["target_date"] <= TRAIN_END]
test = df[df["target_date"] > TRAIN_END]

for name, part in [("train", train), ("test", test)]:
    print(f"{name}: {len(part)} rows | target weeks {part['target_date'].min().date()} to {part['target_date'].max().date()}")

train: 21736 rows | target weeks 2007-12-29 to 2023-12-30
test: 3614 rows | target weeks 2024-01-06 to 2026-08-29


In [12]:
pd.DataFrame({
    "train": train["risk_next"].value_counts(normalize=True),
    "test": test["risk_next"].value_counts(normalize=True),
}).loc[["low", "medium", "high"]].round(2)

,train,test
risk_next,,
low,0.75,0.53
medium,0.15,0.30
high,0.10,0.16


Training comes out at about 75% low / 15% medium / 10% high, as expected from the percentile rules. The test period has many more medium and high weeks (30% / 16%) because of the large 2024-2026 outbreaks. The thresholds are fixed from past data, so the test labels show how unusual recent weeks are compared with history; test scores should be read with this shift in mind.

## 10. Validation checks

The historical features are re-checked independently: for every row, the expected value is looked up in the cleaned data by **date** (district, start date - *k* weeks), instead of re-using the row shifts above. Since every lookup goes back 0 or more weeks, passing these checks also shows that no input feature uses future case counts.

In [13]:
lookup = cleaned.set_index(["district", "start_date"])["cases"]

def cases_weeks_ago(k):
    # cleaned case count k weeks before each row's week (k = -1 means next week)
    keys = pd.MultiIndex.from_arrays([df["district"], df["start_date"] - pd.Timedelta(weeks=k)])
    return lookup.reindex(keys).to_numpy()

# order and uniqueness
assert df.groupby("district")["start_date"].apply(lambda s: s.is_monotonic_increasing).all()
assert not df.duplicated(["district", "start_date"]).any()
assert df[FEATURES].notna().all().all()

# current week and lags (52 lag = 52 observations = 364 days earlier)
assert (df["current_week_cases"].to_numpy() == cases_weeks_ago(0)).all()
for k, col in [(1, "previous_week_cases"), (2, "previous_2_weeks_cases"),
               (3, "previous_3_weeks_cases"), (52, "seasonal_cases_previous_52_weeks")]:
    assert (df[col].to_numpy() == cases_weeks_ago(k)).all(), col

# moving averages use exactly 4 / 8 weeks: the current week and the ones before it
assert np.allclose(df["moving_average_4_weeks"], np.mean([cases_weeks_ago(k) for k in range(4)], axis=0))
assert np.allclose(df["moving_average_8_weeks"], np.mean([cases_weeks_ago(k) for k in range(8)], axis=0))
assert (df["weekly_case_change"] == df["current_week_cases"] - df["previous_week_cases"]).all()

# targets are the following week
assert (df["target_date"] - df["start_date"] == pd.Timedelta(days=7)).all()
assert (df["cases_next"].to_numpy() == cases_weeks_ago(-1)).all()
assert np.allclose(df["target_log_next"], np.log1p(df["cases_next"]))

# chronological split, no shared target weeks
assert len(train) + len(test) == len(df)
assert train["target_date"].max() < test["target_date"].min()

# thresholds come from training-period weeks only
check = cleaned[cleaned["start_date"] <= TRAIN_END].groupby("district")["cases"].quantile([0.75, 0.90]).unstack()
assert np.allclose(thresholds.to_numpy(), check.loc[thresholds.index].to_numpy())

# features are in case units: whole, non-negative counts; averages within the observed range
count_cols = ["current_week_cases"] + LAG_COLS
assert (df[count_cols] >= 0).all().all()
assert all(pd.api.types.is_integer_dtype(df[c]) for c in count_cols)
assert df[["moving_average_4_weeks", "moving_average_8_weeks"]].ge(0).all().all()
assert df[["moving_average_4_weeks", "moving_average_8_weeks"]].le(cleaned["cases"].max()).all().all()

print("all checks passed")

all checks passed


In [14]:
print("features:", len(FEATURES))
for name, part in [("train", train), ("test", test)]:
    print(f"{name}: {part.shape[0]} rows | input weeks {part['start_date'].min().date()} to {part['start_date'].max().date()}"
          f" | target weeks {part['target_date'].min().date()} to {part['target_date'].max().date()}")

print("missing values (features + targets):")
pd.DataFrame({"train": train[FEATURES + TARGETS].isna().sum(), "test": test[FEATURES + TARGETS].isna().sum()})

features: 11
train: 21736 rows | input weeks 2007-12-22 to 2023-12-23 | target weeks 2007-12-29 to 2023-12-30
test: 3614 rows | input weeks 2023-12-30 to 2026-08-22 | target weeks 2024-01-06 to 2026-08-29
missing values (features + targets):


,train,test
current_week_cases,0,0
previous_week_cases,0,0
previous_2_weeks_cases,0,0
previous_3_weeks_cases,0,0
seasonal_cases_previous_52_weeks,0,0
moving_average_4_weeks,0,0
moving_average_8_weeks,0,0
weekly_case_change,0,0
seasonal_week_sin,0,0
seasonal_week_cos,0,0


## 11. Save processed datasets

Each file contains identifiers and dates, the 11 input features, the three targets and the data-quality flags. `current_week_cases` is the actual (cleaned) case count for the input week.

In [15]:
OUT_COLS = [
    "district", "start_date", "target_date", "year", "week",
    *FEATURES,
    "cases_next", "target_log_next", "risk_next",
    "is_imputed", "dup_week", "target_imputed",
]

train[OUT_COLS].sort_values(["start_date", "district"]).to_csv(TRAIN_PATH, index=False, date_format="%Y-%m-%d")
test[OUT_COLS].sort_values(["start_date", "district"]).to_csv(TEST_PATH, index=False, date_format="%Y-%m-%d")
thresholds.to_csv(THRESH_PATH)

print("saved", TRAIN_PATH.name, train[OUT_COLS].shape, "|", TEST_PATH.name, test[OUT_COLS].shape, "|", THRESH_PATH.name, thresholds.shape)

saved train.csv (21736, 22) | test.csv (3614, 22) | 03_risk_thresholds.csv (26, 2)


## 12. Summary

| Group | Features | Concept |
|---|---|---|
| Current observation | `current_week_cases` | value at week *t* |
| Lags | `previous_week_cases`, `previous_2_weeks_cases`, `previous_3_weeks_cases` | 1st-3rd order lags |
| Seasonal lag | `seasonal_cases_previous_52_weeks` | value 52 weeks earlier |
| Moving averages | `moving_average_4_weeks`, `moving_average_8_weeks` | short / medium-term level |
| Change | `weekly_case_change` | 1st-order difference |
| Calendar season | `seasonal_week_sin`, `seasonal_week_cos` | cyclical encoding of the day-of-year position |
| Location | `district_code` | categorical identifier |

**Targets:** `cases_next` and `target_log_next` (regression, original and log scale) and `risk_next` (low / medium / high from each district's training-period 75th and 90th percentiles - relative categories, not official outbreak thresholds).

**Split:** by target week - train up to 31 Dec 2023, test from 1 Jan 2024. No shuffling, no overlapping target weeks.

**Files:** `train.csv`, `test.csv` and `03_risk_thresholds.csv` (used later to turn predicted counts into the same risk levels).

**Things to keep in mind for modelling**
- All case features are in raw units; whether a log-scale target works better than raw counts is left to model evaluation.
- Rows flagged with `is_imputed`, `dup_week` or `target_imputed` can be excluded when scoring.
- The test period contains more medium/high weeks than training (recent large outbreaks), and the last few test weeks (2026 w34-36) may be under-reported (see notebook 02).